In [1]:
import os

# base libraries
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import matplotlib.colors as mcolors
import seaborn as sns

import math
from math import sqrt

from scipy import stats
from scipy.stats import ttest_ind, pointbiserialr, mannwhitneyu

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder 
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy, SparseCategoricalCrossentropy
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam

from datetime import datetime, timedelta

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.utils.validation")

In [3]:
# Adjust the path to match your Box sync folder location
box_path = os.path.expanduser(r"C:\Users\HIALAB\Box\Human_AGV_project\ISU_Modeling\Code")

In [5]:
# Specify the directory and filename
save_dir = "./Images/"

In [7]:
data = pd.read_csv(os.path.join(box_path, 'Per_Interaction_Data_Merged.csv'))
data

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,AGV_Path_Complexity,Trust_before,Cross_First_Flag,User_Relative_Speed,AGV_Relative_Speed,Frechet_Distance_3,Frechet_Distance_5,Frechet_Distance_7,Frechet_Distance_10,Cross_First
0,1,High,2024-5-3,14,28,53,1,1900-01-01 15:02:48,1900-01-01 15:03:38,219,...,Straight,4.9,True,74.580213,100.420508,730.13,1188.92,1532.83,1771.61,N\A
1,1,High,2024-5-3,14,28,53,2,1900-01-01 15:03:47,1900-01-01 15:04:29,0,...,Complex,10.0,True,47.888087,232.345812,14.56,97.05,108.21,247.34,N\A
2,1,High,2024-5-3,14,28,53,3,1900-01-01 15:04:37,1900-01-01 15:05:34,268,...,Complex,10.0,True,63.130599,120.176130,282.09,482.54,821.32,1067.08,User
3,1,High,2024-5-3,14,28,53,4,1900-01-01 15:05:46,1900-01-01 15:06:37,779,...,Complex,10.0,True,43.571998,263.284416,421.34,544.15,577.55,610.84,User
4,1,High,2024-5-3,14,28,53,5,1900-01-01 15:06:41,1900-01-01 15:07:44,868,...,Complex,7.0,True,93.720008,91.614207,486.65,852.85,1225.51,1722.14,N\A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1317,46,Low,2024-8-26,16,55,4,12,1900-01-01 17:07:02,1900-01-01 17:08:21,595,...,Complex,9.0,NaN,77.121444,90.961763,224.51,507.80,789.37,1176.65,User
1318,46,Low,2024-8-26,16,55,4,13,1900-01-01 17:08:24,1900-01-01 17:09:05,0,...,Complex,9.0,NaN,54.297007,286.165319,420.27,627.75,714.38,937.14,AGV
1319,46,Low,2024-8-26,16,55,4,14,1900-01-01 17:09:08,1900-01-01 17:09:52,0,...,Straight,10.0,NaN,82.319060,221.787091,408.09,668.13,919.97,1180.07,User
1320,46,Low,2024-8-26,16,55,4,15,1900-01-01 17:09:56,1900-01-01 17:10:31,67,...,Straight,10.0,NaN,65.826092,278.817960,301.57,562.78,774.23,998.19,User


In [9]:
# Convert 'StartTime' and 'EndTime' to datetime format if they are not already
data['StartTime'] = pd.to_datetime(data['StartTime'])
data['EndTime'] = pd.to_datetime(data['EndTime'])

# Create a new column 'duration' as the difference between 'EndTime' and 'StartTime'
data['Duration'] = (data['EndTime'] - data['StartTime']).dt.total_seconds()

In [11]:
null_counts = data.isnull().sum()
null_counts

PID                        0
DRate                      0
date                       0
hr                         0
min                        0
s                          0
AGVname                    0
StartTime                  0
EndTime                    0
GazeDuration               0
mean_dist                  0
min_dist                   0
max_dist                   0
std_dist                   0
mean_agv_spd               0
min_agv_spd                0
max_agv_spd                0
std_spd                    0
task                       0
Trust                      0
Safe                       0
Comfort                    0
Expect                     0
Age                        0
Gender                     0
Ethnicity                  0
GamingFrequency            0
VRExperience               0
VRHeadsetExperience        0
AGVInteraction             0
PerfectAutomation          0
TrustPropensity            0
AutomationExperience       0
Trust1                     0
Trust2        

In [13]:
data.drop(columns=['Cross_First_Flag'], inplace=True)

In [15]:
study_data = pd.read_csv(os.path.join(box_path, 'study_data_processed.csv'))
study_data

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,AGV_Yaw,AGV_Roll,AGV_spd,Timestamp,AGVname,PID,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance
0,5051.235,8610.830,-209.725,-21.416662,-92.196247,-0.159610,60.524000,-107.387000,156.022000,4990.711200,...,89.999444,0.002393,0.000238,2024-10-25 15:02:48.900,1,1,High,0.0,0.000000,6812.026513
1,5051.235,8610.830,-209.725,-21.168361,-92.384752,0.290698,60.493000,-108.133857,156.034571,4990.742143,...,89.999339,-0.000460,0.154715,2024-10-25 15:02:49.000,1,1,High,0.0,84.241150,6811.521389
2,5051.235,8610.830,-209.725,-21.240655,-92.761660,0.339485,60.271857,-108.963714,156.053429,4990.963429,...,89.999565,-0.001784,0.526054,2024-10-25 15:02:49.100,1,1,High,0.0,74.712591,6809.601293
3,5051.235,8610.830,-209.725,-21.004733,-93.120707,0.251285,59.831714,-109.665143,156.194286,4991.403286,...,90.000198,-0.001954,0.893006,2024-10-25 15:02:49.200,1,1,High,0.0,35.834089,6806.150079
4,5051.235,8610.830,-209.725,-20.717933,-91.225685,0.512156,59.295143,-110.065286,156.430571,4991.940286,...,90.001157,-0.001464,1.263035,2024-10-25 15:02:49.300,1,1,High,0.0,66.574460,6801.168018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6848366,4814.045,8544.847,-209.725,9.361614,-83.257447,2.434928,-57.158167,-36.919167,162.815000,4871.203000,...,56.090021,-0.034390,0.000980,2024-10-25 17:11:16.500,16,46,Low,0.0,0.237071,3572.199178
6848367,4814.045,8544.847,-209.725,8.837808,-85.161383,1.814595,-57.379333,-36.885333,162.748167,4871.424000,...,56.092403,-0.034348,0.001081,2024-10-25 17:11:16.600,16,46,Low,0.0,0.228819,3572.182128
6848368,4814.045,8544.847,-209.725,8.142000,-88.676587,0.913597,-57.698500,-36.836667,162.646667,4871.743000,...,56.094889,-0.034428,0.000787,2024-10-25 17:11:16.700,16,46,Low,0.0,0.239264,3572.164349
6848369,4814.045,8544.847,-209.725,7.388442,-92.010121,0.207441,-57.935833,-36.714167,162.534667,4871.980333,...,56.097355,-0.034399,0.000931,2024-10-25 17:11:16.800,16,46,Low,0.0,0.242956,3572.146282


In [16]:
null_counts = study_data.isnull().sum()
null_counts

User_X                 0
User_Y                 0
User_Z                 0
User_Pitch             0
User_Yaw               0
User_Roll              0
U_X                    0
U_Y                    0
U_Z                    0
GazeOrigin_X           0
GazeOrigin_Y           0
GazeOrigin_Z           0
GazeDirection_X        0
GazeDirection_Y        0
GazeDirection_Z        0
Confidence             0
Gaze_on_AGV            0
AGV_X                  0
AGV_Y                  0
AGV_Z                  0
AGV_Pitch              0
AGV_Yaw                0
AGV_Roll               0
AGV_spd                0
Timestamp              0
AGVname                0
PID                    0
DRate                  0
User_Relative_Speed    0
AGV_Relative_Speed     0
AGV_User_distance      0
dtype: int64

In [17]:
eye_target_data = pd.read_csv(os.path.join(box_path, 'eye_target_data.csv'))
eye_target_data

,PID,DRate,AGVname,Timestamp,EyeTarget
0,1,High,1.0,15:02:48,BP_Affordance_C_0
1,1,High,1.0,15:02:49,BP_Affordance_C_0
2,1,High,1.0,15:02:50,BP_CheckPoint3
3,1,High,1.0,15:02:51,NW_FactoryFloor61
4,1,High,1.0,15:02:52,NW_FactoryFloor67
...,...,...,...,...,...
683373,46,Low,16.0,17:11:12,BP_CheckPoint3
683374,46,Low,16.0,17:11:13,BP_CheckPoint3
683375,46,Low,16.0,17:11:14,BP_CheckPoint3
683376,46,Low,16.0,17:11:15,BP_CheckPoint3


In [18]:
null_counts = eye_target_data.isnull().sum()
null_counts

PID             0
DRate           0
AGVname      2640
Timestamp       0
EyeTarget    4869
dtype: int64

In [19]:
eye_target_data = eye_target_data.dropna(subset=['AGVname'])

In [20]:
print(eye_target_data.columns.tolist())

['PID', 'DRate', 'AGVname', 'Timestamp', 'EyeTarget']


In [21]:
study_data['Timestamp'] = pd.to_datetime(study_data['Timestamp'])
eye_target_data['Timestamp'] = pd.to_datetime(eye_target_data['Timestamp'])

C:\Users\HIALAB\AppData\Local\Temp\ipykernel_99964\3599962841.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  eye_target_data['Timestamp'] = pd.to_datetime(eye_target_data['Timestamp'])
C:\Users\HIALAB\AppData\Local\Temp\ipykernel_99964\3599962841.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eye_target_data['Timestamp'] = pd.to_datetime(eye_target_data['Timestamp'])


In [22]:
# Filter rows where milliseconds are 900
study_data_by_sec = study_data[study_data['Timestamp'].dt.microsecond == 000000]

# Remove milliseconds from the timestamp
study_data_by_sec['Timestamp'] = pd.to_datetime(study_data_by_sec['Timestamp']).dt.strftime('%H:%M:%S')
study_data_by_sec

C:\Users\HIALAB\AppData\Local\Temp\ipykernel_99964\3359013595.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  study_data_by_sec['Timestamp'] = pd.to_datetime(study_data_by_sec['Timestamp']).dt.strftime('%H:%M:%S')


,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,AGV_Yaw,AGV_Roll,AGV_spd,Timestamp,AGVname,PID,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance
1,5051.235000,8610.830000,-209.725,-21.168361,-92.384752,0.290698,60.493000,-108.133857,156.034571,4990.742143,...,89.999339,-0.000460,0.154715,15:02:49,1,1,High,0.000000,84.241150,6811.521389
11,5053.666571,8615.704000,-209.725,-18.911368,-30.124304,-1.673294,62.059714,-93.105857,158.717429,4991.595857,...,90.013001,0.000188,3.943791,15:02:50,1,1,High,5.503666,178.675513,6725.458832
21,5056.453857,8621.069857,-209.725,-12.435596,29.756961,0.000200,66.965429,-83.693714,160.744857,4989.152000,...,90.031424,-0.000391,7.750412,15:02:51,1,1,High,17.531309,377.153393,6476.594544
31,5052.985000,8623.575000,-209.725,-8.044145,54.297518,-1.040257,69.836429,-57.435143,159.692000,4977.797250,...,90.048880,-0.001023,11.951223,15:02:52,1,1,High,0.000000,584.436719,6056.983221
41,4981.397857,8586.533714,-209.725,-31.997808,47.980614,-3.696765,77.184857,-52.889143,151.637857,4906.009000,...,90.065508,-0.000745,14.977350,15:02:53,1,1,High,194.812458,758.046672,5393.609440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6848321,4814.045000,8544.847000,-209.725,7.603790,-93.517554,0.112428,-57.683000,-37.738571,162.621143,4871.727857,...,55.960986,-0.034329,0.000901,17:11:12,16,46,Low,0.000000,0.313207,3573.126479
6848331,4814.045000,8544.847000,-209.725,9.593631,-101.241813,-2.028508,-59.239429,-37.026714,162.851143,4873.284286,...,55.989912,-0.034332,0.001034,17:11:13,16,46,Low,0.000000,0.299045,3572.918260
6848341,4814.045000,8544.847000,-209.725,10.106079,-86.682990,-1.234411,-57.745714,-36.808571,162.943429,4871.790429,...,56.019236,-0.034376,0.000893,17:11:14,16,46,Low,0.000000,0.309687,3572.707442
6848351,4814.045000,8544.847000,-209.725,10.802710,-103.167978,-1.137901,-59.808714,-36.247000,163.049286,4873.853429,...,56.048648,-0.034306,0.000943,17:11:15,16,46,Low,0.000000,0.313157,3572.496169


In [23]:
# Drop rows where PID is in [5, 18, 23, 25]
data = data[~data['PID'].isin([5, 18, 23, 25])]
study_data_by_sec = study_data_by_sec[~study_data_by_sec['PID'].isin([5, 18, 23, 25])]
eye_target_data = eye_target_data[~eye_target_data['PID'].isin([5, 18, 23, 25])]

In [24]:
# Ensure consistent types
data['PID'] = data['PID'].astype(int)
data['DRate'] = data['DRate'].astype(str)
data['AGVname'] = data['AGVname'].astype(int)

study_data_by_sec['PID'] = study_data_by_sec['PID'].astype(int)
study_data_by_sec['DRate'] = study_data_by_sec['DRate'].astype(str)
study_data_by_sec['AGVname'] = study_data_by_sec['AGVname'].astype(int)
study_data_by_sec['Timestamp'] = pd.to_datetime(study_data_by_sec['Timestamp']).dt.time

eye_target_data['PID'] = eye_target_data['PID'].astype(int)
eye_target_data['DRate'] = eye_target_data['DRate'].astype(str)
eye_target_data['AGVname'] = eye_target_data['AGVname'].astype(int)
eye_target_data['Timestamp'] = pd.to_datetime(eye_target_data['Timestamp'], format='%H:%M:%S')
eye_target_data['Timestamp'] = pd.to_datetime(eye_target_data['Timestamp']).dt.time

C:\Users\HIALAB\AppData\Local\Temp\ipykernel_99964\522005595.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  study_data_by_sec['Timestamp'] = pd.to_datetime(study_data_by_sec['Timestamp']).dt.time


In [25]:
eye_target_data

,PID,DRate,AGVname,Timestamp,EyeTarget
0,1,High,1,15:02:48,BP_Affordance_C_0
1,1,High,1,15:02:49,BP_Affordance_C_0
2,1,High,1,15:02:50,BP_CheckPoint3
3,1,High,1,15:02:51,NW_FactoryFloor61
4,1,High,1,15:02:52,NW_FactoryFloor67
...,...,...,...,...,...
683373,46,Low,16,17:11:12,BP_CheckPoint3
683374,46,Low,16,17:11:13,BP_CheckPoint3
683375,46,Low,16,17:11:14,BP_CheckPoint3
683376,46,Low,16,17:11:15,BP_CheckPoint3


In [26]:
study_data_by_sec

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,AGV_Yaw,AGV_Roll,AGV_spd,Timestamp,AGVname,PID,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance
1,5051.235000,8610.830000,-209.725,-21.168361,-92.384752,0.290698,60.493000,-108.133857,156.034571,4990.742143,...,89.999339,-0.000460,0.154715,15:02:49,1,1,High,0.000000,84.241150,6811.521389
11,5053.666571,8615.704000,-209.725,-18.911368,-30.124304,-1.673294,62.059714,-93.105857,158.717429,4991.595857,...,90.013001,0.000188,3.943791,15:02:50,1,1,High,5.503666,178.675513,6725.458832
21,5056.453857,8621.069857,-209.725,-12.435596,29.756961,0.000200,66.965429,-83.693714,160.744857,4989.152000,...,90.031424,-0.000391,7.750412,15:02:51,1,1,High,17.531309,377.153393,6476.594544
31,5052.985000,8623.575000,-209.725,-8.044145,54.297518,-1.040257,69.836429,-57.435143,159.692000,4977.797250,...,90.048880,-0.001023,11.951223,15:02:52,1,1,High,0.000000,584.436719,6056.983221
41,4981.397857,8586.533714,-209.725,-31.997808,47.980614,-3.696765,77.184857,-52.889143,151.637857,4906.009000,...,90.065508,-0.000745,14.977350,15:02:53,1,1,High,194.812458,758.046672,5393.609440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6848321,4814.045000,8544.847000,-209.725,7.603790,-93.517554,0.112428,-57.683000,-37.738571,162.621143,4871.727857,...,55.960986,-0.034329,0.000901,17:11:12,16,46,Low,0.000000,0.313207,3573.126479
6848331,4814.045000,8544.847000,-209.725,9.593631,-101.241813,-2.028508,-59.239429,-37.026714,162.851143,4873.284286,...,55.989912,-0.034332,0.001034,17:11:13,16,46,Low,0.000000,0.299045,3572.918260
6848341,4814.045000,8544.847000,-209.725,10.106079,-86.682990,-1.234411,-57.745714,-36.808571,162.943429,4871.790429,...,56.019236,-0.034376,0.000893,17:11:14,16,46,Low,0.000000,0.309687,3572.707442
6848351,4814.045000,8544.847000,-209.725,10.802710,-103.167978,-1.137901,-59.808714,-36.247000,163.049286,4873.853429,...,56.048648,-0.034306,0.000943,17:11:15,16,46,Low,0.000000,0.313157,3572.496169


In [27]:
study_data_by_sec['EyeTarget'] = None

study_data_by_sec = study_data_by_sec.merge(
    eye_target_data[['PID', 'DRate', 'AGVname', 'Timestamp', 'EyeTarget']],
    on=['PID', 'DRate', 'AGVname', 'Timestamp'],
    how='left',  # Keeps all rows from per_interaction_data_merged
    suffixes=('', '_new')
)

study_data_by_sec['EyeTarget'] = study_data_by_sec['EyeTarget_new']

# Drop the temporary column
study_data_by_sec.drop(columns=['EyeTarget_new'], inplace=True)

In [28]:
print(study_data_by_sec.columns.tolist())

['User_X', 'User_Y', 'User_Z', 'User_Pitch', 'User_Yaw', 'User_Roll', 'U_X', 'U_Y', 'U_Z', 'GazeOrigin_X', 'GazeOrigin_Y', 'GazeOrigin_Z', 'GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z', 'Confidence', 'Gaze_on_AGV', 'AGV_X', 'AGV_Y', 'AGV_Z', 'AGV_Pitch', 'AGV_Yaw', 'AGV_Roll', 'AGV_spd', 'Timestamp', 'AGVname', 'PID', 'DRate', 'User_Relative_Speed', 'AGV_Relative_Speed', 'AGV_User_distance', 'EyeTarget']


In [29]:
null_counts = study_data_by_sec.isnull().sum()
null_counts

User_X                   0
User_Y                   0
User_Z                   0
User_Pitch               0
User_Yaw                 0
User_Roll                0
U_X                      0
U_Y                      0
U_Z                      0
GazeOrigin_X             0
GazeOrigin_Y             0
GazeOrigin_Z             0
GazeDirection_X          0
GazeDirection_Y          0
GazeDirection_Z          0
Confidence               0
Gaze_on_AGV              0
AGV_X                    0
AGV_Y                    0
AGV_Z                    0
AGV_Pitch                0
AGV_Yaw                  0
AGV_Roll                 0
AGV_spd                  0
Timestamp                0
AGVname                  0
PID                      0
DRate                    0
User_Relative_Speed      0
AGV_Relative_Speed       0
AGV_User_distance        0
EyeTarget              585
dtype: int64

In [30]:
study_data_by_sec = study_data_by_sec.dropna(subset=['EyeTarget'])

In [31]:
study_data_by_sec.shape

(7376313, 32)

In [32]:
# Filter and count only EyeTarget values that contain 'AGV'
agv_targets = eye_target_data['EyeTarget'][eye_target_data['EyeTarget'].astype(str).str.contains('AGV', na=False)]

# Get counts for those
agv_target_counts = agv_targets.value_counts()

print(agv_target_counts)

EyeTarget
AGV_Sphere4     10929
AGV_Sphere3      9359
AGV_Sphere2      9151
AGV_Sphere6      8410
AGV_Sphere7      6551
AGV_Sphere9      4683
AGV_Sphere8      4559
AGV_Sphere5      4330
AGV_Sphere11     3772
AGV_Sphere10     3426
AGV_Sphere12     2463
AGV_Sphere       2065
AGV_Sphere14     1327
AGV_Sphere13     1039
AGV_Sphere16      558
AGV_Sphere15      554
Name: count, dtype: int64


In [35]:
# Project vectors to XY plane
vec_user_to_agv_2d = np.stack([
    study_data_by_sec['AGV_X'] - study_data_by_sec['User_X'],
    study_data_by_sec['AGV_Y'] - study_data_by_sec['User_Y']
], axis=1)

gaze_direction_2d = np.stack([
    study_data_by_sec['GazeDirection_X'],
    study_data_by_sec['GazeDirection_Y']
], axis=1)

def normalize_vectors(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.where(norms == 0, 1, norms)  # Avoid division by zero

vec_user_to_agv_norm_2d = normalize_vectors(vec_user_to_agv_2d)
gaze_direction_norm_2d = normalize_vectors(gaze_direction_2d)

# Compute angles of each vector (relative to +X axis)
theta_agv = np.arctan2(vec_user_to_agv_norm_2d[:, 1], vec_user_to_agv_norm_2d[:, 0])
theta_gaze = np.arctan2(gaze_direction_norm_2d[:, 1], gaze_direction_norm_2d[:, 0])

# Compute the difference in angles
angles_rad = theta_gaze - theta_agv

# Normalize to [0, 2π]
angles_rad = (angles_rad + 2 * np.pi) % (2 * np.pi)

# Convert to degrees
angles_deg_360 = np.degrees(angles_rad) 

# Add to the dataframe
study_data_by_sec['Angle_to_AGV'] = angles_deg_360

# Keep the FOV flag (using 95-degree threshold)
study_data_by_sec['AGV_in_FOV'] = angles_deg_360 <= 60

# Preview the new columns
study_data_by_sec[['Timestamp', 'Angle_to_AGV', 'AGV_in_FOV']].head()

,Timestamp,Angle_to_AGV,AGV_in_FOV
0,15:02:49,203.080077,False
1,15:02:49,203.080077,False
2,15:02:49,203.080077,False
3,15:02:49,203.080077,False
4,15:02:49,203.080077,False


In [37]:
study_data_by_sec['Angle_to_AGV'].describe()

count    7.376313e+06
mean     1.891156e+02
std      1.160065e+02
min      4.414765e-04
25%      7.707186e+01
50%      2.123235e+02
75%      2.918066e+02
max      3.599997e+02
Name: Angle_to_AGV, dtype: float64

In [39]:
# Copy dataset to avoid changes to the original one
destination_data = study_data_by_sec.copy()

# Create an empty list to store angles and FOV flags
angles_to_dest = []
dest_in_fov_flags = []

# Group the data
grouped = destination_data.groupby(['PID', 'DRate', 'AGVname'])

# Iterate over each group
for (pid, drate, agvname), group in grouped:
    group = group.sort_values(by='Timestamp').reset_index(drop=True)
    
    # Step 1: Get destination coordinates from the last row
    dest_x = group.iloc[-1]['User_X']
    dest_y = group.iloc[-1]['User_Y']
    dest_z = group.iloc[-1]['User_Z']
    
    # Step 2: Compute vectors from current position to destination for each row
    vec_user_to_dest = np.stack([
        dest_x - group['User_X'],
        dest_y - group['User_Y'],
        dest_z - group['User_Z']
    ], axis=1)
    
    # Step 3: Gaze direction vectors
    gaze_direction = np.stack([
        group['GazeDirection_X'],
        group['GazeDirection_Y'],
        group['GazeDirection_Z']
    ], axis=1)
    
    # Step 4: Normalize vectors
    def normalize_vectors(vectors):
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        return vectors / np.where(norms == 0, 1, norms)
    
    vec_user_to_dest_norm = normalize_vectors(vec_user_to_dest)
    gaze_direction_norm = normalize_vectors(gaze_direction)
    
    # Step 5: Calculate dot product and angle
    cos_theta = np.sum(vec_user_to_dest_norm * gaze_direction_norm, axis=1)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    angles_deg = np.degrees(np.arccos(cos_theta))
    
    # Step 6: Determine if destination is in FOV (<= 95 degrees)
    dest_in_fov = angles_deg <= 95
    
    # Append results to the list
    angles_to_dest.extend(angles_deg)
    dest_in_fov_flags.extend(dest_in_fov)

# Add results to the dataframe
destination_data['Angle_to_Destination'] = angles_to_dest
destination_data['Destination_in_FOV'] = dest_in_fov_flags

# Preview results
destination_data[['PID', 'DRate', 'AGVname', 'Timestamp', 'Angle_to_Destination', 'Destination_in_FOV']].head()

,PID,DRate,AGVname,Timestamp,Angle_to_Destination,Destination_in_FOV
0,1,High,1,15:02:49,112.737321,False
1,1,High,1,15:02:49,112.737321,False
2,1,High,1,15:02:49,112.737321,False
3,1,High,1,15:02:49,112.737321,False
4,1,High,1,15:02:49,112.737321,False


In [41]:
study_data_by_sec['Angle_to_Destination'] = destination_data['Angle_to_Destination']
study_data_by_sec['Destination_in_FOV'] = destination_data['Destination_in_FOV']

study_data_by_sec['Angle_to_Destination'].describe()

count    7.376313e+06
mean     5.525440e+01
std      3.915203e+01
min      1.030160e-01
25%      1.770384e+01
50%      5.133093e+01
75%      9.000000e+01
max      1.791138e+02
Name: Angle_to_Destination, dtype: float64

In [45]:
study_data_by_sec.head()

,User_X,User_Y,User_Z,User_Pitch,User_Yaw,User_Roll,U_X,U_Y,U_Z,GazeOrigin_X,...,PID,DRate,User_Relative_Speed,AGV_Relative_Speed,AGV_User_distance,EyeTarget,Angle_to_AGV,AGV_in_FOV,Angle_to_Destination,Destination_in_FOV
0,5051.235,8610.83,-209.725,-21.168361,-92.384752,0.290698,60.493,-108.133857,156.034571,4990.742143,...,1,High,0.0,84.24115,6811.521389,BP_Affordance_C_0,203.080077,False,112.737321,False
1,5051.235,8610.83,-209.725,-21.168361,-92.384752,0.290698,60.493,-108.133857,156.034571,4990.742143,...,1,High,0.0,84.24115,6811.521389,BP_Affordance_C_0,203.080077,False,112.737321,False
2,5051.235,8610.83,-209.725,-21.168361,-92.384752,0.290698,60.493,-108.133857,156.034571,4990.742143,...,1,High,0.0,84.24115,6811.521389,BP_Affordance_C_0,203.080077,False,112.737321,False
3,5051.235,8610.83,-209.725,-21.168361,-92.384752,0.290698,60.493,-108.133857,156.034571,4990.742143,...,1,High,0.0,84.24115,6811.521389,BP_Affordance_C_0,203.080077,False,112.737321,False
4,5051.235,8610.83,-209.725,-21.168361,-92.384752,0.290698,60.493,-108.133857,156.034571,4990.742143,...,1,High,0.0,84.24115,6811.521389,BP_Affordance_C_0,203.080077,False,112.737321,False


In [49]:
file_name = "study_data_by_sec_processed.csv"  
full_path = os.path.join(box_path, file_name)

study_data_by_sec.to_csv(full_path, index=False)

In [ ]:
# Step 1: Randomly pick 6 PID and AGVname pairs
random_groups = study_data_by_sec[['PID', 'AGVname']].drop_duplicates().sample(6, random_state=2021)

# Create subplots: 6 groups x 2 DRate conditions (High & Low)
fig, axes = plt.subplots(6, 2, figsize=(18, 30), sharey=True)

# Step 2: Loop through each random group
for group_idx, (_, row) in enumerate(random_groups.iterrows()):
    random_pid = row['PID']
    random_agv = row['AGVname']
    
    print(f"Selected PID: {random_pid}, AGVname: {random_agv}")

    # Filter the data for this group
    group_data = study_data_by_sec[
        (study_data_by_sec['PID'] == random_pid) &
        (study_data_by_sec['AGVname'] == random_agv)
    ]

    # Step 3: Create a plot for DRate = High and DRate = Low
    for drate_idx, drate in enumerate(['High', 'Low']):
        ax1 = axes[group_idx, drate_idx]  # Primary axis (left Y-axis)
        
        # Filter the data for the current DRate
        drate_data = group_data[group_data['DRate'] == drate].copy()

        if drate_data.empty:
            ax1.set_title(f"PID {random_pid}, AGV {random_agv}, DRate {drate} (No Data)")
            ax1.axis('off')
            continue
        
        # Sort data (optional but good practice)
        drate_data = drate_data.sort_values(by='Timestamp').reset_index(drop=True)

        # X-axis: sequential entry number (time step)
        x_values = drate_data.index + 1
        
        # Y-axis 1: Gaze Angle (raw degrees)
        y_gaze_angle = drate_data['Gaze_Angle_Degrees']

        # Y-axis 2: AGV_User_distance
        y_agv_user_dist = drate_data['AGV_User_distance']

        # Plot Gaze Angle on primary axis
        line1 = ax1.plot(x_values, y_gaze_angle, color='blue', linestyle='-', label='Gaze Angle')
        ax1.set_ylim(0, 361)
        ax1.set_ylabel('Gaze Angle (Degrees)', color='blue', fontsize=10)
        ax1.set_xlabel('Entry Number', fontsize=10)
        ax1.grid(True)

        ax1.axhspan(0, 60, facecolor='red', alpha=0.1)
        ax1.axhspan(300, 360, facecolor='red', alpha=0.1)

        # Secondary Y-axis for AGV_User_distance
        ax2 = ax1.twinx()
        line2 = ax2.plot(x_values, y_agv_user_dist, color='green', linestyle='--', label='AGV_User_distance')
        ax2.set_ylabel('AGV_User_distance', color='green', fontsize=10)

        # Combine legends
        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper right', fontsize=8)

        # Title for the subplot
        ax1.set_title(f'PID {random_pid}, AGV {random_agv}, DRate {drate}', fontsize=12)

plt.tight_layout(pad=4.0)
filename = f'Gaze_Angle_and_AGV-User_Distance_Over_Time.png'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)
plt.show()

In [ ]:
# Step 1: Randomly pick 3 PID and AGVname pairs
random_groups = study_data_by_sec[['PID', 'AGVname']].drop_duplicates().sample(3, random_state=2025)

# Step 2: Create subplots - 3 groups x 2 DRate conditions (High & Low)
fig, axes = plt.subplots(3, 2, figsize=(14, 12), subplot_kw={'projection': 'polar'})

# Step 3: Loop through each random group
for group_idx, (_, row) in enumerate(random_groups.iterrows()):
    random_pid = row['PID']
    random_agv = row['AGVname']
    
    print(f"Selected PID: {random_pid}, AGVname: {random_agv}")

    # Filter the data for this group
    group_data = study_data_by_sec[
        (study_data_by_sec['PID'] == random_pid) &
        (study_data_by_sec['AGVname'] == random_agv)
    ]

    # Step 4: Create polar plots for both DRate = High and Low
    for drate_idx, drate in enumerate(['High', 'Low']):
        ax = axes[group_idx, drate_idx]  # Polar axis for this group/DRate
        
        # Filter the data for the current DRate
        drate_data = group_data[group_data['DRate'] == drate].copy()

        if drate_data.empty:
            ax.set_title(f"PID {random_pid}, AGV {random_agv}, DRate {drate} (No Data)")
            ax.axis('off')
            continue
        
        # Sort data by Timestamp for consistent index progression
        drate_data = drate_data.sort_values(by='Timestamp').reset_index(drop=True)

        # Limit to the first 24 points
        # drate_data = drate_data.head(1000)

        # Theta = Gaze_Angle_Degrees converted to radians
        theta_rad = np.radians(drate_data['Gaze_Angle_Degrees'])

        # r = AGV_User_distance
        r = drate_data['AGV_User_distance']

        # Color based on index (time progression)
        color_by_index = np.linspace(0, 1, len(drate_data))  # Gradient from 0 to 1

        # Scatter plot in polar coordinates
        points = ax.scatter(
            theta_rad, r,
            c=color_by_index,            # Color by index progression
            cmap='viridis',              # Choose colormap (or try 'plasma', 'cool', etc.)
            alpha=0.75
        )

        # Add colorbar only for the first column (optional)
        if drate_idx == 1:
            cbar = plt.colorbar(points, ax=ax, orientation='vertical', pad=0.1)
            cbar.set_label('Index Progression')

        # Shade the area between 95° and 265°
        theta_start_deg = 60
        theta_end_deg = 300
        theta_start_rad = np.radians(theta_start_deg)
        theta_end_rad = np.radians(theta_end_deg)
        theta_width_rad = theta_end_rad - theta_start_rad

        # Set title for each polar plot
        ax.set_title(f'PID {random_pid}, AGV {random_agv}, DRate {drate}', fontsize=12)

        # Set r limits to standardize across subplots
        ax.set_ylim(0, study_data_by_sec['AGV_User_distance'].max())

        # Polar plot settings
        ax.set_theta_zero_location("N")  # Zero degrees is at the top
        ax.set_theta_direction(-1)       # Angles increase clockwise

# Layout and show the plots
plt.tight_layout(pad=2.0)
filename = f'AGV_User_Distance_and_Angle_Over_Time_Polar_Representation.png'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)
plt.show()

In [ ]:
# Filter for PID 35, AGV1, DRate High
target_pid = 35
target_agvname = 1  # Assuming AGV1 is stored as 1, adjust if needed
target_drate = 'High'

# Filter your dataframe
filtered_data = study_data_by_sec[
    (study_data_by_sec['PID'] == target_pid) &
    (study_data_by_sec['AGVname'] == target_agvname) &
    (study_data_by_sec['DRate'] == target_drate)
].copy()

# Select the first 24 entries (or fewer if not enough)
sample_data = filtered_data.head(24).reset_index(drop=True)

# Debugging: print how many points you have
print(f"Number of entries selected: {len(sample_data)}")

# Set up subplots - 4 rows x 6 columns (24 total plots)
fig, axes = plt.subplots(4, 6, figsize=(18, 12))  
axes = axes.flatten()

# Loop through each entry
for idx, row in sample_data.iterrows():
    
    ax = axes[idx]
    
    # User position
    user_pos = np.array([row['User_X'], row['User_Y']])
    
    # AGV position
    agv_pos = np.array([row['AGV_X'], row['AGV_Y']])
    
    # Gaze direction (XY-plane)
    gaze_dir = np.array([row['GazeDirection_X'], row['GazeDirection_Y']])
    
    # Normalize gaze direction for consistent arrow length
    gaze_dir_norm = gaze_dir / (np.linalg.norm(gaze_dir) + 1e-6)
    
    # Arrow length for plotting
    arrow_len = 10000.0
    gaze_arrow = gaze_dir_norm * arrow_len
    
    # Compute FOV boundary angles (+/- 95 degrees)
    angle_rad = np.arctan2(gaze_dir_norm[1], gaze_dir_norm[0])  # In radians
    
    left_bound_angle_rad = angle_rad + np.radians(60)
    right_bound_angle_rad = angle_rad - np.radians(60)
    
    # Convert to degrees for the wedge
    angle_deg = np.degrees(angle_rad)
    left_bound_angle_deg = angle_deg + 60
    right_bound_angle_deg = angle_deg - 60
    
    # Create FOV wedge
    fov_wedge = Wedge(
        center=(user_pos[0], user_pos[1]),
        r=arrow_len,
        theta1=right_bound_angle_deg,
        theta2=left_bound_angle_deg,
        facecolor='green',
        alpha=0.2
    )
    ax.add_patch(fov_wedge)
    
    # Compute vectors for boundary arrows
    left_bound_vec = np.array([np.cos(left_bound_angle_rad), np.sin(left_bound_angle_rad)]) * arrow_len
    right_bound_vec = np.array([np.cos(right_bound_angle_rad), np.sin(right_bound_angle_rad)]) * arrow_len
    
    # Plot AGV position
    ax.scatter(agv_pos[0], agv_pos[1], color='red', label='AGV')
    
    # Plot user position
    ax.scatter(user_pos[0], user_pos[1], color='blue', label='User')
    
    # Plot gaze direction arrow
    ax.arrow(user_pos[0], user_pos[1], gaze_arrow[0], gaze_arrow[1], 
             head_width=0.3, head_length=0.3, fc='green', ec='green', label='Gaze Direction')
    
    # Plot FOV boundary arrows (dashed)
    ax.arrow(user_pos[0], user_pos[1], left_bound_vec[0], left_bound_vec[1], 
             linestyle='dashed', color='blue', alpha=0.6)
    
    ax.arrow(user_pos[0], user_pos[1], right_bound_vec[0], right_bound_vec[1], 
             linestyle='dashed', color='blue', alpha=0.6)
    
    # Title showing AGV_in_FOV status
    title = f"AGV in FOV: {row['AGV_in_FOV']}"
    ax.set_title(title)
    
    # Consistent axis limits (centered on user position)
    ax.set_xlim(user_pos[0] - 10500, user_pos[0] + 10500)
    ax.set_ylim(user_pos[1] - 10500, user_pos[1] + 10500)
    
    ax.set_aspect('equal')
    
    # Show grid and axis labels
    ax.grid(True)
    ax.axis('on')

# If you want to add a legend
axes[-1].legend(loc='lower right')

# Layout adjustment
plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
study_data_by_sec['AGV_in_FOV'].value_counts()

In [ ]:
from matplotlib.patches import Wedge

# Pick 24 random entries
sample_data = study_data_by_sec.sample(6, random_state=42).reset_index()

# Set up subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))  # 4 rows x 6 columns
axes = axes.flatten()

# Loop through each random entry
for idx, row in sample_data.iterrows():
    
    ax = axes[idx]
    
    # User position
    user_pos = np.array([row['User_X'], row['User_Y']])
    
    # AGV position
    agv_pos = np.array([row['AGV_X'], row['AGV_Y']])
    
    # Gaze direction (project onto XY-plane)
    gaze_dir = np.array([row['GazeDirection_X'], row['GazeDirection_Y']])
    
    # Normalize gaze direction for consistent arrow length
    gaze_dir_norm = gaze_dir / (np.linalg.norm(gaze_dir) + 1e-6)
    
    # Length of arrows for plotting
    arrow_len = 10000.0
    gaze_arrow = gaze_dir_norm * arrow_len
    
    # Compute FOV boundary angles (+/- 95 degrees)
    angle_rad = np.arctan2(gaze_dir_norm[1], gaze_dir_norm[0])  # In radians
    
    left_bound_angle_rad = angle_rad + np.radians(60)
    right_bound_angle_rad = angle_rad - np.radians(60)
    
    # Convert to degrees for Wedge
    angle_deg = np.degrees(angle_rad)
    left_bound_angle_deg = angle_deg + 60
    right_bound_angle_deg = angle_deg - 60
    
    # Create FOV wedge (matplotlib uses degrees and counterclockwise)
    fov_wedge = Wedge(
        center=(user_pos[0], user_pos[1]),
        r=arrow_len,
        theta1=right_bound_angle_deg,
        theta2=left_bound_angle_deg,
        facecolor='green',
        alpha=0.2  # Transparency for FOV
    )
    ax.add_patch(fov_wedge)
    
    # Compute vectors for boundaries
    left_bound_vec = np.array([np.cos(left_bound_angle_rad), np.sin(left_bound_angle_rad)]) * arrow_len
    right_bound_vec = np.array([np.cos(right_bound_angle_rad), np.sin(right_bound_angle_rad)]) * arrow_len
    
    # Plot AGV position
    ax.scatter(agv_pos[0], agv_pos[1], color='red', label='AGV')
    
    # Plot user position
    ax.scatter(user_pos[0], user_pos[1], color='blue', label='User')
    
    # Plot gaze direction arrow
    ax.arrow(user_pos[0], user_pos[1], gaze_arrow[0], gaze_arrow[1], 
             head_width=0.3, head_length=0.3, fc='green', ec='green', label='Gaze Direction')
    
    # Plot FOV boundary dashed arrows
    ax.arrow(user_pos[0], user_pos[1], left_bound_vec[0], left_bound_vec[1], 
             linestyle='dashed', color='blue', alpha=0.6)
    
    ax.arrow(user_pos[0], user_pos[1], right_bound_vec[0], right_bound_vec[1], 
             linestyle='dashed', color='blue', alpha=0.6)
    
    # Set title with AGV_in_FOV info
    title = f"AGV in FOV: {row['AGV_in_FOV']}"
    ax.set_title(title)
    
    # Set axis limits to keep plots consistent
    ax.set_xlim(-10000, 25000)
    ax.set_ylim(-10000, 25000)
    
    ax.set_aspect('equal')
    
    # Show grid and axes
    ax.grid(True)
    ax.axis('on')  # Turns axes back on

# Add a legend to the last plot
axes[-1].legend(loc='lower right')

plt.tight_layout()
filename = f'AGV_in_User_Field_of_View_Visualization.png'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)
plt.show()

In [ ]:
study_data_by_sec['AGV_in_FOV_shift'] = study_data_by_sec.groupby('PID')['AGV_in_FOV'].shift()
transitions = study_data_by_sec[study_data_by_sec['AGV_in_FOV'] != study_data_by_sec['AGV_in_FOV_shift']]
print(f"Number of FOV transitions: {len(transitions)}")

In [ ]:
study_data_by_sec.groupby('AGV_in_FOV')['User_Relative_Speed'].describe()

In [ ]:
result = ttest_ind(
    study_data_by_sec[study_data_by_sec['AGV_in_FOV']]['AGV_User_distance'],
    study_data_by_sec[~study_data_by_sec['AGV_in_FOV']]['AGV_User_distance']
)

# Extract and format the p-value and t-statistic to 3 significant figures
formatted_t = f"{result.statistic:.5g}"
formatted_p = f"{result.pvalue:.10g}"

print(f"T-statistic: {formatted_t}")
print(f"P-value: {formatted_p}")

In [ ]:
study_data_by_sec.groupby('AGV_in_FOV')['AGV_User_distance'].describe()

In [ ]:
plt.figure(figsize=(12, 8)) 

sns.histplot(
    data=study_data_by_sec,
    x='AGV_User_distance',
    hue='AGV_in_FOV',
    kde=True
)

plt.title("AGV_User_distance when AGV is In vs Out of FOV", fontsize=16)
plt.xlabel("AGV_User_distance", fontsize=14)
plt.ylabel("Count", fontsize=14)

plt.grid(True)
filename = f'AGV_User_Distance_Distribution_By_FOV_Status.png'
plt.savefig(save_dir + filename, bbox_inches = 'tight', pad_inches=0.1)
plt.show()

In [ ]:
grouped_fov = study_data_by_sec.groupby(['PID', 'DRate', 'AGVname'])['AGV_in_FOV'].mean().reset_index()

# Rename column to clarify meaning
grouped_fov.rename(columns={'AGV_in_FOV': 'FOV_Percentage'}, inplace=True)

# Preview the result
grouped_fov.head()

In [ ]:
data['FOV_Percentage'] = None  # This triggers the suffixing

data = data.merge(
    grouped_fov[['PID', 'DRate', 'AGVname', 'FOV_Percentage']],
    on=['PID', 'DRate', 'AGVname'],
    how='left',
    suffixes=('', '_new')
)

# Now it makes sense to do:
data['FOV_Percentage'] = data['FOV_Percentage_new']
data.drop(columns=['FOV_Percentage_new'], inplace=True)

In [ ]:
data.head()

In [ ]:
print(data.columns.tolist())

In [ ]:
null_counts = data.isnull().sum()
null_counts

In [ ]:
# Create a grouping based on median FOV_Percentage
median_fov = data['FOV_Percentage'].median()
print(median_fov)

data['High_FOV'] = data['FOV_Percentage'] > median_fov

# Merge back subjective measures (Trust, etc.) at the same group level if they exist
# Assuming you already have a grouped dataset with Trust, Safe, Comfort, Expect

# Perform t-tests for each subjective variable
columns_to_test = ['Trust', 'Safe', 'Comfort', 'Expect']

for col in columns_to_test:
    result = ttest_ind(
        data[data['High_FOV']][col].dropna(),
        data[~data['High_FOV']][col].dropna()
    )
    
    formatted_t = f"{result.statistic:.3g}"
    formatted_p = f"{result.pvalue:.3g}"
    
    print(f"Column: {col}")
    print(f"  T-statistic: {formatted_t}")
    print(f"  P-value:     {formatted_p}")
    print("-" * 40)

In [ ]:
cross_first_data = data[data['Cross_First'].isin(['AGV', 'User'])]
# cross_first_data = cross_first_data.drop(['Cross_First_Flag', 'Trust_before'], axis=1)
cross_first_data['Cross_First'] = np.where(cross_first_data['Cross_First'] == 'User', 0, 1)
# cross_first_data = cross_first_data.dropna(subset=['Trust_before'])
cross_first_data.head()

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(x='Cross_First', y='FOV_Percentage', data=cross_first_data)
plt.title('FOV_Percentage by Cross_First Decision')
plt.xlabel('Cross_First')
plt.ylabel('FOV_Percentage')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(cross_first_data['FOV_Percentage'], kde=True, bins=20)
plt.title('Distribution of FOV_Percentage')
plt.xlabel('FOV_Percentage')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
group_0 = cross_first_data[cross_first_data['Cross_First'] == 0]['FOV_Percentage']
group_1 = cross_first_data[cross_first_data['Cross_First'] == 1]['FOV_Percentage']

# Perform Mann-Whitney U test
u_stat, p_value = mannwhitneyu(group_0, group_1, alternative='two-sided')

# Display results
print(f"Mann-Whitney U statistic: {u_stat:.3f}")
print(f"P-value: {p_value:.3g}")

In [ ]:
corr, pval = pointbiserialr(cross_first_data['Cross_First'], cross_first_data['FOV_Percentage'])

print(f"Point-Biserial Correlation: {corr:.3g}, P-value: {pval:.3g}")

In [ ]:
# Summary statistics for the gaze direction columns
study_data_by_sec[['GazeDirection_X', 'GazeDirection_Y', 'GazeDirection_Z']].head

In [ ]:
unique_groups = (
    study_data_by_sec
    .groupby(['AGVname', 'PID'])
    .size()
    .reset_index()
    .rename(columns={0: 'count'})
)

# Step 2: Randomly select 3 groups
sample_groups = unique_groups.sample(6, random_state=2025)

num_groups = len(sample_groups)

# Step 3: Compute Global Limits for All Plots
x_min = study_data_by_sec['GazeDirection_X'].min()
x_max = study_data_by_sec['GazeDirection_X'].max()
y_min = study_data_by_sec['GazeDirection_Y'].min()
y_max = study_data_by_sec['GazeDirection_Y'].max()

# Square limits
global_min = min(x_min, y_min)
global_max = max(x_max, y_max)

# Step 4: Create subplots for each group: 2 subplots per group (High, Low)
fig, axes = plt.subplots(num_groups, 2, figsize=(12, 6 * num_groups))

# Ensure axes is always 2D
axes = axes if num_groups > 1 else [axes]
axes = np.array(axes).reshape(num_groups, 2)

# Step 5: Iterate through each sampled group
for idx, row in enumerate(sample_groups.itertuples(index=False)):
    agv = row.AGVname
    pid = row.PID

    # Filter for this AGVname and PID
    subset = study_data_by_sec[
        (study_data_by_sec['AGVname'] == agv) & 
        (study_data_by_sec['PID'] == pid)
    ]

    # Step 6: Plot for both DRate conditions: High and Low
    for i, drate in enumerate(['High', 'Low']):
        ax = axes[idx, i]

        # Filter for the current DRate
        drate_subset = subset[subset['DRate'] == drate].reset_index(drop=True)

        if drate_subset.empty:
            ax.set_title(f"AGV {agv}, PID {pid}, DRate {drate}\n(No Data)", fontsize=14)
            ax.axis('off')
            continue

        # Create the gradient of colors (light to dark)
        color_values = np.linspace(0.9, 0.2, len(drate_subset))  # Reverse gradient
        colors = plt.cm.Greens(color_values)

        # Scatter plot
        ax.scatter(
            drate_subset['GazeDirection_X'],
            drate_subset['GazeDirection_Y'],
            c=colors,
            s=50
        )

        # Inside your plot loop:
        ax.set_xlim(global_min - 1, global_max + 1)
        ax.set_ylim(global_min - 1, global_max + 1)

        ax.set_title(f"AGV {agv}, PID {pid}, DRate {drate}", fontsize=14)
        ax.set_xlabel('GazeDirection_X', fontsize=12)
        ax.set_ylabel('GazeDirection_Y', fontsize=12)
        ax.grid(True)
        ax.set_aspect('equal')

# Final layout adjustments
plt.tight_layout(pad=2.0)
plt.show()

### Logistic Regression for High_FOV

In [ ]:
selected_columns_df = ['User_Trajectory', 'AGV_Approaching', 'AGV_User_Combination', 'DRate', 'Duration',
       'GazeDuration', 'mean_dist', 'min_dist', 'max_dist',
       'mean_agv_spd', 'min_agv_spd', 'max_agv_spd', 'Cross_First', 'User_Relative_Speed',
       'Frechet_Distance_10', 'High_FOV']

numerical_columns = ['Duration',
       'GazeDuration', 'mean_dist', 'min_dist', 'max_dist',
       'mean_agv_spd', 'min_agv_spd', 'max_agv_spd','User_Relative_Speed',
       'Frechet_Distance_10']

categorical_columns = ['User_Trajectory', 'AGV_Approaching', 'AGV_User_Combination', 'DRate', 'Cross_First']

X1 = cross_first_data[selected_columns_df]
X1[1000:1100]

In [ ]:
# Apply one-hot encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = encoder.fit_transform(X1[categorical_columns])

# Add encoded columns back to the DataFrame
encoded_columns = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_data, columns=encoded_columns, index=X.index)
encoded_X1 = pd.concat([X1.drop(columns=categorical_columns), encoded_df], axis=1)
encoded_X1[1000:1200]

In [ ]:
# Shuffle the dataset
shuffled_X1 = encoded_X1.sample(frac=1, random_state=42).reset_index(drop=True)

# Define the split indices
n = len(shuffled_X1)
train_data = shuffled_X1[0:int(n * 0.7)]
val_data = shuffled_X1[int(n * 0.7):int(n * 0.9)]
test_data = shuffled_X1[int(n * 0.9):]

In [ ]:
# Compute normalization statistics for numerical columns
train_mean = train_data[numerical_columns].mean()
train_std = train_data[numerical_columns].std()

# Normalize numerical columns only, while keeping other columns intact
train_data[numerical_columns] = (train_data[numerical_columns] - train_mean) / train_std
val_data[numerical_columns] = (val_data[numerical_columns] - train_mean) / train_std
test_data[numerical_columns] = (test_data[numerical_columns] - train_mean) / train_std

val_data.head()

In [ ]:
# Separate features (X) and target variable (y)
X_train = train_data.drop(columns=['High_FOV'])
y_train = train_data['High_FOV']

X_val = val_data.drop(columns=['High_FOV'])
y_val = val_data['High_FOV']

X_test = test_data.drop(columns=['High_FOV'])
y_test = test_data['High_FOV']

# Verify the shapes of the sets
print(f"Train set: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Validation set: X_val {X_val.shape}, y_val {y_val.shape}")
print(f"Test set: X_test {X_test.shape}, y_test {y_test.shape}")

In [ ]:
# Initialize logistic regression model
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Train on training data
log_reg.fit(X_train, y_train)

# Predict on validation data
y_val_pred = log_reg.predict(X_val)

# Evaluate performance on validation set
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_accuracy:.3f}")

# More metrics (optional)
print("Validation Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("Validation Classification Report:")
print(classification_report(y_val, y_val_pred))

In [ ]:
# Predict on test data
y_test_pred = log_reg.predict(X_test)

# Evaluate performance on test set
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.3f}")

# More metrics
print("Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("Test Classification Report:")
print(classification_report(y_test, y_test_pred))

In [ ]:
from sklearn.metrics import roc_curve

# Probability estimates for positive class
y_test_probs = log_reg.predict_proba(X_test)[:, 1]

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_probs)
auc_score = roc_auc_score(y_test, y_test_probs)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Test Set)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
# Coefficients array
coefficients = log_reg.coef_[0]  # shape: (n_features,)

# Create a DataFrame for easy viewing
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': coefficients
})

# Add absolute value of coefficients for sorting by impact
coef_df['abs_coef'] = coef_df['Coefficient'].abs()

# Sort by absolute coefficient size
coef_df.sort_values(by='abs_coef', ascending=False, inplace=True)

# Display
print(coef_df[['Feature', 'Coefficient']])

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [50, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best RF params:", grid_search.best_params_)
print("Best RF CV accuracy:", grid_search.best_score_)

In [ ]:
scores = cross_val_score(RandomForestClassifier(random_state=42), X_train, y_train, cv=5)
print("Cross-validated accuracy:", scores.mean())

### Linear Regression for FOV_Percentage

In [ ]:
selected_columns_df = ['PID','User_Trajectory', 'AGV_Approaching', 'AGV_User_Combination', 'DRate', 'Duration',
       'GazeDuration', 'mean_dist', 'min_dist', 'max_dist',
       'mean_agv_spd', 'min_agv_spd', 'max_agv_spd', 'Cross_First', 'User_Relative_Speed',
       'Frechet_Distance_10', 'FOV_Percentage']

numerical_columns = ['Duration',
       'GazeDuration', 'mean_dist', 'min_dist', 'max_dist',
       'mean_agv_spd', 'min_agv_spd', 'max_agv_spd','User_Relative_Speed',
       'Frechet_Distance_10']

categorical_columns = ['User_Trajectory', 'AGV_Approaching', 'AGV_User_Combination', 'DRate', 'Cross_First']

X2 = cross_first_data[selected_columns_df]
X2[1000:1100]

In [ ]:
# Apply one-hot encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = encoder.fit_transform(X2[categorical_columns])

# Add encoded columns back to the DataFrame
encoded_columns = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_data, columns=encoded_columns, index=X.index)
encoded_X2 = pd.concat([X2.drop(columns=categorical_columns), encoded_df], axis=1)
encoded_X2[1000:1200]

In [ ]:
column_indices = {name: i for i, name in enumerate(encoded_X2.columns)}

n = len(encoded_X2)
train_data = encoded_X2[0:int(n*0.7)]
val_data = encoded_X2[int(n*0.7):int(n*0.9)]
test_data = encoded_X2[int(n*0.9):]

num_features = encoded_X2.shape[1]

In [ ]:
encoded_X2.head()

#### Normalization

In [ ]:
# Compute normalization statistics for numerical columns
train_mean = train_data[numerical_columns].mean()
train_std = train_data[numerical_columns].std()

# Normalize numerical columns only, while keeping other columns intact
train_data[numerical_columns] = (train_data[numerical_columns] - train_mean) / train_std
val_data[numerical_columns] = (val_data[numerical_columns] - train_mean) / train_std
test_data[numerical_columns] = (test_data[numerical_columns] - train_mean) / train_std

val_data.head()

In [ ]:
class WindowGenerator():
  def __init__(self, input_width=None, label_width=None, shift=None,
               train_df=train_data, val_df=val_data, test_df=test_data,
               label_columns=None):
    # Store the raw data.
    self.train_df = train_data
    self.val_df = val_data
    self.test_df = test_data

    # Work out the label column indices.
    self.label_columns = label_columns
    if label_columns is not None:
      self.label_columns_indices = {name: i for i, name in
                                    enumerate(label_columns)}
    self.column_indices = {name: i for i, name in
                           enumerate(train_df.columns)}

    # Work out the window parameters.
    self.input_width = input_width
    self.label_width = label_width
    self.shift = shift

    self.total_window_size = input_width + shift

    self.input_slice = slice(0, input_width)
    self.input_indices = np.arange(self.total_window_size)[self.input_slice]

    self.label_start = self.total_window_size - self.label_width
    self.labels_slice = slice(self.label_start, None)
    self.label_indices = np.arange(self.total_window_size)[self.labels_slice]

  def __repr__(self):
    return '\n'.join([
        f'Total window size: {self.total_window_size}',
        f'Input indices: {self.input_indices}',
        f'Label indices: {self.label_indices}',
        f'Label column name(s): {self.label_columns}'])

In [ ]:
# Example datasets (train_data, val_data, test_data)
window = WindowGenerator(
    input_width=8,
    label_width=1,
    shift=1,
    train_df=train_data,
    val_df=val_data,
    test_df=test_data,
    label_columns=['FOV_Percentage']
)
window

In [ ]:
def split_window(self, features):
  inputs = features[:, self.input_slice, :]
  labels = features[:, self.labels_slice, :]
  if self.label_columns is not None:
    labels = tf.stack(
        [labels[:, :, self.column_indices[name]] for name in self.label_columns],
        axis=-1)

  # Slicing doesn't preserve static shape information, so set the shapes
  # manually. This way the `tf.data.Datasets` are easier to inspect.
  inputs.set_shape([None, self.input_width, None])
  labels.set_shape([None, self.label_width, None])

  # print(f'the inputs are', inputs)
  # print(f'the labels are', labels)

  return inputs, labels

WindowGenerator.split_window = split_window

In [ ]:
# Stack three slices, the length of the total window.
example_window = tf.stack([np.array(train_data[:window.total_window_size]),
                           np.array(train_data[100:100+window.total_window_size]),
                           np.array(train_data[200:200+window.total_window_size])])

print(f'the example window is:', example_window)

example_inputs, example_labels = window.split_window(example_window)

print('All shapes are: (batch, time, features)')
print(f'Window shape: {example_window.shape}')
print(f'Inputs shape: {example_inputs.shape}')
print(f'Labels shape: {example_labels.shape}')

In [ ]:
window.example = example_inputs, example_labels

In [ ]:
import matplotlib.ticker as ticker
def plot(self, model=None, plot_col='FOV_Percentage', fullpath=None, max_subplots=3, title=''):
    inputs, labels = self.example
    plt.figure(figsize=(12, 10))

    plot_col_index = self.column_indices[plot_col]
    max_n = min(max_subplots, len(inputs))
    for n in range(max_n):
        plt.subplot(max_n, 1, n + 1)
        plt.ylabel(f'{plot_col}')
        plt.plot(self.input_indices, inputs[n, :, plot_col_index],
                 label='Inputs', marker='.', zorder=-10)

        if self.label_columns:
            label_col_index = self.label_columns_indices.get(plot_col, None)
        else:
            label_col_index = plot_col_index

        if label_col_index is None:
            continue

        plt.scatter(self.label_indices, labels[n, :, label_col_index],
                    edgecolors='k', label='Labels', c='#2ca02c', s=64)
        if model is not None:
            predictions = model(inputs)
            plt.scatter(self.label_indices, predictions[n, :, label_col_index],
                        marker='X', edgecolors='k', label='Predictions',
                        c='#ff7f0e', s=64)

        if n == 0:
            plt.legend()

        plt.ylim(0, 1)
        plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    plt.xlabel('Trial')

    # Add a title if provided
    if title:
        plt.suptitle(title, fontsize=16)

    # Save the plot if a filename is provided
    if fullpath:
        if not fullpath.endswith('.png'):  # Add default extension if not provided
            fullpath += '.png'
        # plt.savefig(fullpath, bbox_inches='tight')
        print(f"Plot saved as {fullpath}")

    plt.show()

WindowGenerator.plot = plot

In [ ]:
window.plot(plot_col='FOV_Percentage', fullpath=None, title='Later Labels of FOV_Percentage; Width = 8')

In [ ]:
def make_dataset(self, data):
    """
    Create a dataset of sliding windows, ensuring that windows are created
    within the same PID group.
    """
    # Convert to NumPy array for compatibility
    data = np.array(data, dtype=np.float32)
    # print(f"Data converted to NumPy array with shape: {data.shape}")

    # Identify the PID column index
    pid_index = self.column_indices.get('PID', None)
    if pid_index is None:
        raise ValueError("PID column not found in data.")
    # print(f"PID column index: {pid_index}")

    # Split data by PID groups
    datasets = []
    unique_pids = np.unique(data[:, pid_index])  # Get unique PIDs
    # print(f"Unique PIDs found: {unique_pids}")

    for pid in unique_pids:
        pid_data = data[data[:, pid_index] == pid]
        
        # Skip if not enough data for a sequence
        if len(pid_data) < self.total_window_size:
            print(f"Skipping PID {pid} due to insufficient data ({len(pid_data)} rows)")
            continue
        
        ds = tf.keras.utils.timeseries_dataset_from_array(
            data=pid_data,
            targets=None,
            sequence_length=self.total_window_size,
            sequence_stride=1,
            shuffle=True,
            batch_size=32,
        )
        # print(f"Dataset for PID {pid} created.")

        # Apply the split_window method to extract inputs and labels
        ds = ds.map(self.split_window)
        # print(f"Applied split_window for PID {pid}.")
        datasets.append(ds)

    # Concatenate all datasets
    # print("Concatenating datasets for all PIDs.")
    full_dataset = datasets[0]
    for ds in datasets[1:]:
        full_dataset = full_dataset.concatenate(ds)

    print("All datasets concatenated successfully.")
    print()
    print(f"the full dataset is:", full_dataset)
    return full_dataset

WindowGenerator.make_dataset = make_dataset

In [ ]:
@property
def train(self):
  return self.make_dataset(self.train_df)

@property
def val(self):
  return self.make_dataset(self.val_df)

@property
def test(self):
  return self.make_dataset(self.test_df)

@property
def example(self):
  """Get and cache an example batch of `inputs, labels` for plotting."""
  result = getattr(self, '_example', None)
  if result is None:
    # No example batch was found, so get one from the `.train` dataset
    result = next(iter(self.train))
    # And cache it for next time
    self._example = result
  return result

WindowGenerator.train = train
WindowGenerator.val = val
WindowGenerator.test = test
WindowGenerator.example = example

In [ ]:
# Each element is an (inputs, label) pair.
window.train.element_spec
window.val.element_spec
window.test.element_spec

In [ ]:
for example_inputs, example_labels in window.train.take(1):
  print(f'Inputs shape (batch, time, features): {example_inputs.shape}')
  print(f'Labels shape (batch, time, features): {example_labels.shape}')

In [ ]:
single_step_window = WindowGenerator(
    input_width=1, label_width=1, shift=1,
    label_columns=['FOV_Percentage'])
single_step_window

In [ ]:
# Separate features (X) and target variable (y)
X_train = train_data.drop(columns=['FOV_Percentage'])
y_train = train_data['FOV_Percentage']

X_val = val_data.drop(columns=['FOV_Percentage'])
y_val = val_data['FOV_Percentage']

X_test = test_data.drop(columns=['FOV_Percentage'])
y_test = test_data['FOV_Percentage']

# Verify the shapes of the sets
print(f"Train set: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Validation set: X_val {X_val.shape}, y_val {y_val.shape}")
print(f"Test set: X_test {X_test.shape}, y_test {y_test.shape}")

In [ ]:
val_performance = {}
performance = {}

In [ ]:
class Baseline(tf.keras.Model):
  def __init__(self, label_index=None):
    super().__init__()
    self.label_index = label_index

  def call(self, inputs):
    if self.label_index is None:
      return inputs
    result = inputs[:, :, self.label_index]
    return result[:, :, tf.newaxis]

baseline = Baseline(label_index=column_indices['FOV_Percentage'])

baseline.compile(loss=tf.keras.losses.MeanSquaredError(), metrics=[tf.keras.metrics.MeanAbsoluteError()])

val_performance['Baseline'] = baseline.evaluate(single_step_window.val, return_dict=True)
performance['Baseline'] = baseline.evaluate(single_step_window.test, verbose=0, return_dict=True)

In [ ]:
wide_width = 8

wide_window = WindowGenerator(
    input_width=wide_width, label_width=wide_width, shift=1,
    label_columns=['FOV_Percentage']
)

print('Input shape:', wide_window.example[0].shape)
print('Output shape:', baseline(wide_window.example[0]).shape)

# Dynamically update the title with the correct width
wide_window.plot(baseline, fullpath=None, title=f'Baseline; Width = {wide_width}')

In [ ]:
linear = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1)
])

In [ ]:
print('Input shape:', single_step_window.example[0].shape)
print('Output shape:', linear(single_step_window.example[0]).shape)

In [ ]:
MAX_EPOCHS = 10

def compile_and_fit(model, window, patience=2):
  early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                    patience=patience,
                                                    mode='min')

  model.compile(loss=tf.keras.losses.MeanSquaredError(),
                optimizer=tf.keras.optimizers.Adam(),
                metrics=[tf.keras.metrics.MeanAbsoluteError()])

  history = model.fit(window.train, epochs=MAX_EPOCHS,
                      validation_data=window.val,
                      callbacks=[early_stopping])
  return history

In [ ]:
history = compile_and_fit(linear, single_step_window)

val_performance['Linear'] = linear.evaluate(single_step_window.val, return_dict=True)
performance['Linear'] = linear.evaluate(single_step_window.test, verbose=0, return_dict=True)

In [ ]:
all_numerical_columns = numerical_columns + ['FOV_Percentage']  

# Extract the corresponding weights from the linear model
weights = linear.layers[0].kernel[:len(all_numerical_columns), 0].numpy()

# Plot the weights
plt.figure(figsize=(10, 6))
plt.bar(x=range(len(all_numerical_columns)), height=weights)

# Set axis labels
axis = plt.gca()
axis.set_xticks(range(len(all_numerical_columns)))
axis.set_xticklabels(all_numerical_columns, rotation=90)

# Title and save the figure
plt.title("Weights for Numerical Columns", fontsize=16)
# plt.savefig('/content/drive/My Drive/weights_plot_for_linear_model_with_width_one.png', format="png", bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
dense = tf.keras.Sequential([
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=1)
])

history = compile_and_fit(dense, single_step_window)

val_performance['Dense'] = dense.evaluate(single_step_window.val, return_dict=True)
performance['Dense'] = dense.evaluate(single_step_window.test, verbose=0, return_dict=True)

In [ ]:
for window_width in range(5, 10):
    print(f"\n Running model with Window Width = {window_width}\n")

    dense_window = WindowGenerator(
        input_width=window_width,
        label_width=1,
        shift=1,
        label_columns=['FOV_Percentage']
    )

    print(f"\n The dense window is = {dense_window}\n")

    multi_step_dense = tf.keras.Sequential([
    # Shape: (time, features) => (time*features)
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=8, activation='relu'),
    tf.keras.layers.Dense(units=4, activation='relu'),
    tf.keras.layers.Dense(units=2, activation='relu'),
    tf.keras.layers.Dense(units=1),
    # Add back the time dimension.
    # Shape: (outputs) => (1, outputs)
    tf.keras.layers.Reshape([1, -1]),])

    print('Input shape:', dense_window.example[0].shape)
    print('Output shape:', multi_step_dense(dense_window.example[0]).shape)

    history = compile_and_fit(multi_step_dense, dense_window)

    val_performance[f'Multi step Dense - Window {window_width}'] = multi_step_dense.evaluate(dense_window.val, return_dict=True)
    performance[f'Multi step Dense - Window {window_width}'] = multi_step_dense.evaluate(dense_window.test, verbose=0, return_dict=True)

print("\n All models trained and evaluated. Performance results:")
print("\n Validation Performance:")
for key, value in val_performance.items():
    print(f"{key}: {value}")

print("\n Test Performance:")
for key, value in performance.items():
    print(f"{key}: {value}")

In [ ]:
for conv_width in range(1, 5):
    print(f"\nRunning Conv1D model with Conv Window Width = {conv_width}\n")

    conv_window = WindowGenerator(
        input_width=conv_width,
        label_width=1,
        shift=1,
        label_columns=['FOV_Percentage']
    )

    print(f"\nThe conv window is = {conv_window}\n")

    print("Conv model on `conv_window`")
    print('Input shape:', conv_window.example[0].shape)

    conv_model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(filters=32,
                               kernel_size=(conv_width,),
                               activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=1)
    ])

    print('Output shape:', conv_model(conv_window.example[0]).shape)

    history = compile_and_fit(conv_model, conv_window)

    val_performance[f'Conv1D - Window {conv_width}'] = conv_model.evaluate(conv_window.val, return_dict=True)
    performance[f'Conv1D - Window {conv_width}'] = conv_model.evaluate(conv_window.test, verbose=0, return_dict=True)

print("\nAll Conv1D models trained and evaluated. Performance results:")

print("\nValidation Performance:")
for key, value in val_performance.items():
    print(f"{key}: {value}")

print("\nTest Performance:")
for key, value in performance.items():
    print(f"{key}: {value}")

In [ ]:
for wide_width in range(1, 8):
    print(f"\nRunning LSTM model with Window Width = {wide_width}\n")

    wide_window = WindowGenerator(
        input_width=wide_width,
        label_width=1,
        shift=1,
        label_columns=['FOV_Percentage']
    )

    print(f"\nThe wide window is = {wide_window}\n")

    print("LSTM model on `wide_window`")
    print('Input shape:', wide_window.example[0].shape)

    num_features = wide_window.example[0].shape[-1]

    lstm_model = tf.keras.models.Sequential([
        tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(wide_width, num_features)),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=1)
    ])

    print('Output shape:', lstm_model(wide_window.example[0]).shape)

    history = compile_and_fit(lstm_model, wide_window)

    val_performance[f'LSTM - Window {wide_width}'] = lstm_model.evaluate(wide_window.val, return_dict=True)
    performance[f'LSTM - Window {wide_width}'] = lstm_model.evaluate(wide_window.test, verbose=0, return_dict=True)

print("\nAll LSTM models trained and evaluated. Performance results:")

print("\nValidation Performance:")
for key, value in val_performance.items():
    print(f"{key}: {value}")

print("\nTest Performance:")
for key, value in performance.items():
    print(f"{key}: {value}")

In [ ]:
for wide_width in range(1, 8):
    print(f"\nRunning GRU model with Window Width = {wide_width}\n")

    wide_window = WindowGenerator(
        input_width=wide_width,
        label_width=1,
        shift=1,
        label_columns=['FOV_Percentage']
    )

    print(f"\nThe wide window is = {wide_window}\n")

    print("GRU model on `wide_window`")
    print('Input shape:', wide_window.example[0].shape)

    num_features = wide_window.example[0].shape[-1]
    
    gru_model = tf.keras.models.Sequential([
        tf.keras.layers.GRU(64, return_sequences=True, input_shape=(wide_width, num_features)),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        tf.keras.layers.Dense(units=32, activation='relu'),
        # Output layer
        tf.keras.layers.Dense(units=1)
    ])

    print('Output shape:', gru_model(wide_window.example[0]).shape)

    history = compile_and_fit(gru_model, wide_window)

    val_performance[f'GRU - Window {wide_width}'] = gru_model.evaluate(wide_window.val, return_dict=True)
    performance[f'GRU - Window {wide_width}'] = gru_model.evaluate(wide_window.test, verbose=0, return_dict=True)

print("\nAll GRU models trained and evaluated. Performance results:")

print("\nValidation Performance:")
for key, value in val_performance.items():
    print(f"{key}: {value}")

print("\nTest Performance:")
for key, value in performance.items():
    print(f"{key}: {value}")

In [ ]:
categories = ["Baseline", "Linear", "Dense", "Multi step Dense", "Conv1D", "LSTM", "GRU"]

best_models = {}
for category in categories:
    best_model = min(
        {k: v for k, v in val_performance.items() if category in k},
        key=lambda k: val_performance[k]["mean_absolute_error"],
        default=None
    )
    if best_model:
        best_models[best_model] = val_performance[best_model]["mean_absolute_error"]

# Prepare data for plotting
filtered_keys = list(best_models.keys())
val_mae = [val_performance[k]['mean_absolute_error'] for k in filtered_keys]

# Extract test MAE only for available keys in performance
test_mae = [performance[k]['mean_absolute_error'] if k in performance else None for k in filtered_keys]

x_filtered = np.arange(len(filtered_keys))
width = 0.3

# Plot
plt.figure(figsize=(10, 6))
plt.ylabel('MAE [FOV_Percentage]')
plt.bar(x_filtered - 0.17, val_mae, width, label='Validation')

# Only plot test MAE if available
if any(test_mae):
    plt.bar(x_filtered + 0.17, test_mae, width, label='Test')

plt.xticks(ticks=x_filtered, labels=filtered_keys, rotation=90)
plt.legend()

# Save the plot
save_path = '/content/drive/My Drive/performance_best_models.png'
# plt.savefig(save_path, format='png', bbox_inches='tight')
# print(f"Plot saved to {save_path}")

# Display the plot
plt.show()